# Causal Mamba+Motor+변위 예측 (Trial 8 하이퍼파라미터)

### Optuna에서 발견한 Trial 8 결과
```
d_state=32, d_conv=4, expand=1, num_layers=1
lr=2.6e-4, batch_size=128
loss_weight=1000, huber_delta=0.002

30 epoch 결과: SEEN=0.429, UNSEEN=1.703 ★
```

### 본격 학습 (100 epoch)
Trial 8 hyperparameter로 100 epoch 본격 학습 → 최종 성능 확인

### 비교
```
                        SEEN     UNSEEN
속도 예측 (Optuna):     0.523    2.263
변위 예측 (baseline):    0.429    1.809
변위 예측 Trial 8 30ep:  0.429    1.703  ← 30 epoch
변위 예측 Trial 8 100ep: ???      ???    ← 이번 노트북
```

## 1단계: 환경

In [ ]:
import os, sys, pickle, time, warnings
BASE_DIR = os.path.expanduser('~')
DATA_DIR = os.path.join(BASE_DIR, 'blackbird_data')
SAVE_DIR = os.path.join(BASE_DIR, 'causal_mamba_disp_trial8_results')
os.makedirs(SAVE_DIR, exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import numpy as np
import pypose as pp
from scipy.interpolate import interp1d
from scipy.spatial.transform import Rotation, Slerp
from mamba_ssm import Mamba
warnings.filterwarnings('ignore')

torch.cuda.empty_cache()
torch.manual_seed(42); np.random.seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2단계: 모델 (Trial 8 구조)

In [ ]:
class CausalMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
    def forward(self, x): return self.mamba(self.norm(x))

class CNNEncoder(nn.Module):
    def __init__(self, in_ch, c_list=[32, 64], k_list=[7,7], s_list=[3,3], p_list=[3,3]):
        super().__init__()
        layers = []; c_prev = in_ch
        for i in range(len(c_list)):
            layers += [nn.Conv1d(c_prev, c_list[i], k_list[i], stride=s_list[i], padding=p_list[i]),
                       nn.BatchNorm1d(c_list[i]), nn.GELU()]
            c_prev = c_list[i]
        layers.append(nn.Dropout(0.5))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class CausalMambaDispNet(nn.Module):
    def __init__(self, d_state=32, d_conv=4, expand=1, num_layers=1):
        super().__init__()
        self.imu_encoder = CNNEncoder(in_ch=6)
        self.ori_encoder = CNNEncoder(in_ch=3)
        self.motor_encoder = CNNEncoder(in_ch=3)
        self.fcn = nn.Sequential(nn.Linear(192, 64))
        self.bn = nn.BatchNorm1d(64); self.gelu = nn.GELU()
        self.mamba_layers = nn.ModuleList([
            CausalMambaBlock(64, d_state, d_conv, expand) for _ in range(num_layers)
        ])
        self.disp_decoder = nn.Sequential(nn.Linear(64, 128), nn.GELU(), nn.Linear(128, 3))
        self.cov_decoder = nn.Sequential(nn.Linear(64, 128), nn.GELU(), nn.Linear(128, 3))
    def forward(self, acc, gyro, rot_so3, motor):
        imu = torch.cat([acc, gyro], dim=-1)
        x1 = self.imu_encoder(imu.transpose(-1,-2)).transpose(-1,-2)
        x2 = self.ori_encoder(rot_so3.transpose(-1,-2)).transpose(-1,-2)
        x3 = self.motor_encoder(motor.transpose(-1,-2)).transpose(-1,-2)
        x = torch.cat([x1, x2, x3], dim=-1)
        x = self.gelu(self.bn(self.fcn(x).transpose(-1,-2)).transpose(-1,-2))
        for mamba in self.mamba_layers:
            x = x + mamba(x)
        return self.disp_decoder(x), torch.exp(self.cov_decoder(x) - 5.0)

def huber_loss(dist, delta=0.002):
    return F.huber_loss(dist, torch.zeros_like(dist), delta=delta)
def uncertainty_loss(vd, cov):
    return ((vd.pow(2)/cov)+torch.log(cov)).mean()

# Trial 8 하이퍼파라미터
TRIAL8_PARAMS = {
    'd_state': 32,
    'd_conv': 4,
    'expand': 1,
    'num_layers': 1,
    'lr': 0.00025757267464272164,
    'batch_size': 128,
    'loss_weight': 1000.0,
    'huber_delta': 0.002,
}
print(f'Trial 8 하이퍼파라미터:')
for k, v in TRIAL8_PARAMS.items(): print(f'  {k}: {v}')
print(f'\n파라미터: {sum(p.numel() for p in CausalMambaDispNet(**{k:v for k,v in TRIAL8_PARAMS.items() if k in ["d_state","d_conv","expand","num_layers"]}).parameters()):,}')

## 3단계: 데이터 + 변위 라벨

In [ ]:
def load_blackbird(data_path):
    imu_raw = np.loadtxt(os.path.join(data_path, 'imu_data.csv'), delimiter=',')
    gt_raw = np.loadtxt(os.path.join(data_path, 'groundTruthPoses.csv'), delimiter=',')
    thrust_path = os.path.join(data_path, 'thrust_data.csv')
    has_thrust = os.path.exists(thrust_path)
    if has_thrust: thrust_raw = np.loadtxt(thrust_path, delimiter=',')
    R_w_ned = np.array([[1,0,0],[0,-1,0],[0,0,-1.]])
    R_b_i = np.array([[0,-1,0],[1,0,0],[0,0,1.]])
    data = []
    for d in gt_raw:
        ts = d[0]/1e6; t_i = d[1:4]
        R_i = Rotation.from_quat([d[5],d[6],d[7],d[4]]).as_matrix()
        R_it = R_w_ned @ R_i @ R_b_i; t_it = R_w_ned @ t_i + R_it @ np.zeros(3)
        q_it = Rotation.from_matrix(R_it).as_quat()
        data.append([ts, t_it[0], t_it[1], t_it[2], q_it[0], q_it[1], q_it[2], q_it[3]])
    data = np.array(data)
    gt_vel_raw = np.diff(data[:,1:4], axis=0) / np.diff(data[:,0])[:,None]
    gt_vel_raw = np.concatenate([gt_vel_raw[:1], gt_vel_raw], axis=0)
    gt_vel = np.stack([np.convolve(gt_vel_raw[:,i], np.ones(5)/5, 'same') for i in range(3)], axis=1)
    gt_traj = np.concatenate([data, gt_vel], axis=1)
    dt = 0.01
    new_times = np.arange(imu_raw[0,0], imu_raw[-1,0]-dt-0.001, dt)
    gyro = interp1d(imu_raw[:,0], imu_raw[:,1:4], axis=0)(new_times)
    accel = interp1d(imu_raw[:,0], imu_raw[:,4:7], axis=0)(new_times)
    t_start = max(new_times[0], data[0,0]); t_end = min(new_times[-1], data[-1,0])
    if has_thrust:
        motor_interp = interp1d(thrust_raw[:,0], thrust_raw[:,1:4], axis=0, fill_value='extrapolate')(new_times)
        t_start = max(t_start, thrust_raw[0,0]); t_end = min(t_end, thrust_raw[-1,0])
    mask = (new_times >= t_start) & (new_times <= t_end)
    times = new_times[mask]; gyro = gyro[mask]; accel = accel[mask]
    if has_thrust:
        motor = motor_interp[mask]; mm = np.max(np.abs(motor), axis=0) + 1e-8; motor = motor / mm
    else:
        motor = np.zeros((len(times), 3), dtype=np.float32)
    pos = interp1d(gt_traj[:,0], gt_traj[:,1:4], axis=0)(times)
    ori_quat = Slerp(gt_traj[:,0], Rotation.from_quat(gt_traj[:,4:8]))(times).as_quat()
    vel = interp1d(gt_traj[:,0], gt_traj[:,8:11], axis=0)(times)
    return {
        'time': torch.tensor(times, dtype=torch.float64),
        'acc': torch.tensor(accel, dtype=torch.float32),
        'gyro': torch.tensor(gyro, dtype=torch.float32),
        'motor': torch.tensor(motor, dtype=torch.float32),
        'gt_translation': torch.tensor(pos, dtype=torch.float32),
        'gt_orientation': pp.SO3(torch.tensor(ori_quat, dtype=torch.float32)),
        'velocity': torch.tensor(vel, dtype=torch.float32),
    }

S_IDX = 14
STRIDE = 9

class BlackbirdDispDataset(Data.Dataset):
    def __init__(self, data_list, window_size=1000, step_size=3):
        self.windows = []
        for data in data_list:
            seq_len = len(data['acc'])
            pos = data['gt_translation']
            rot = data['gt_orientation']
            for j in range(0, seq_len - window_size - step_size, step_size):
                w_pos = pos[j:j+window_size+STRIDE]
                w_rot = rot[j:j+window_size+STRIDE]
                L_out = (window_size - 1 - 1) // STRIDE + 1
                disps_body = []
                for i in range(L_out):
                    idx_t = S_IDX + i * STRIDE
                    idx_next = idx_t + STRIDE
                    if idx_next < len(w_pos):
                        world_disp = w_pos[idx_next] - w_pos[idx_t]
                        R_t = w_rot[idx_t]
                        body_disp = R_t.Inv() @ world_disp
                        disps_body.append(body_disp)
                if len(disps_body) == 0: continue
                while len(disps_body) < L_out:
                    disps_body.append(disps_body[-1])
                body_disp_tensor = torch.stack(disps_body)
                self.windows.append({
                    'acc': data['acc'][j:j+window_size],
                    'gyro': data['gyro'][j:j+window_size],
                    'motor': data['motor'][j:j+window_size],
                    'gt_rot': data['gt_orientation'][j:j+window_size],
                    'gt_disp': body_disp_tensor,
                })
    def __len__(self): return len(self.windows)
    def __getitem__(self, idx): return self.windows[idx]

def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0].keys()}

SEEN = ['clover/yawForward/maxSpeed5p0', 'halfMoon/yawForward/maxSpeed4p0',
        'star/yawForward/maxSpeed5p0', 'egg/yawForward/maxSpeed8p0',
        'winter/yawForward/maxSpeed4p0']
UNSEEN = ['ampersand/yawForward/maxSpeed2p0', 'sid/yawForward/maxSpeed5p0',
          'oval/yawForward/maxSpeed4p0', 'sphinx/yawForward/maxSpeed4p0',
          'bentDice/yawForward/maxSpeed3p0']

print('데이터 로딩...')
train_data, test_data, eval_data = [], [], []
for traj in SEEN:
    for split in ['train', 'test', 'eval']:
        path = os.path.join(DATA_DIR, split, traj)
        if os.path.exists(path):
            d = load_blackbird(path)
            if d:
                if split == 'train': train_data.append(d)
                elif split == 'test': test_data.append(d)
                else: eval_data.append(d)

unseen_data = []
for traj in UNSEEN:
    path = os.path.join(DATA_DIR, 'eval', traj)
    if os.path.exists(path):
        d = load_blackbird(path)
        if d: unseen_data.append((traj.split('/')[0], d))

print('변위 라벨 계산 중...')
train_dataset = BlackbirdDispDataset(train_data, window_size=1000, step_size=3)
test_dataset = BlackbirdDispDataset(test_data, window_size=1000, step_size=10)
print(f'학습: {len(train_dataset)} | 테스트: {len(test_dataset)}')

## 4단계: Trial 8 본격 학습 (100 epoch)

In [ ]:
best = TRIAL8_PARAMS

net = CausalMambaDispNet(
    d_state=best['d_state'], d_conv=best['d_conv'],
    expand=best['expand'], num_layers=best['num_layers']
).to(device).float()

train_loader = Data.DataLoader(train_dataset, batch_size=best['batch_size'], shuffle=True, collate_fn=collate_fn)
test_loader = Data.DataLoader(test_dataset, batch_size=best['batch_size'], shuffle=False, collate_fn=collate_fn)
optimizer = torch.optim.Adam(net.parameters(), lr=best['lr'], weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.2, patience=5, min_lr=1e-5)

EPOCHS = 100
start = time.time()
best_test = float('inf')
print(f'{"="*70}')
print(f'  Trial 8 본격 학습 시작 (100 epoch)')
print(f'  파라미터: {sum(p.numel() for p in net.parameters()):,}')
print(f'{"="*70}')

for epoch in range(1, EPOCHS+1):
    net.train()
    for batch in train_loader:
        acc = batch['acc'].to(device); gyro = batch['gyro'].to(device)
        motor = batch['motor'].to(device)
        gt_rot = batch['gt_rot'].to(device); gt_disp = batch['gt_disp'].to(device)
        rot_so3 = gt_rot.Log().tensor().float()
        pred_disp, pred_cov = net(acc, gyro, rot_so3, motor)
        L = min(pred_disp.shape[1], gt_disp.shape[1])
        pred_disp = pred_disp[:, :L, :]; gt_disp_match = gt_disp[:, :L, :]
        pred_cov = pred_cov[:, :L, :]
        vd = pred_disp - gt_disp_match
        loss = best['loss_weight']*huber_loss(vd, delta=best['huber_delta']) + 1e-4*uncertainty_loss(vd.detach(), pred_cov)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); optimizer.step()
    net.eval(); ts, nt = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            acc=batch['acc'].to(device); gyro=batch['gyro'].to(device)
            motor=batch['motor'].to(device)
            gt_rot=batch['gt_rot'].to(device); gt_disp=batch['gt_disp'].to(device)
            rot_so3=gt_rot.Log().tensor().float()
            pd,_=net(acc,gyro,rot_so3,motor)
            L = min(pd.shape[1], gt_disp.shape[1])
            pd = pd[:, :L, :]; gd = gt_disp[:, :L, :]
            ts+=torch.sqrt(((pd-gd).norm(dim=-1)**2).mean()).item(); nt+=1
    test_rmse = ts/nt; scheduler.step(test_rmse)
    if test_rmse < best_test:
        best_test = test_rmse
        torch.save(net.state_dict(), os.path.join(SAVE_DIR, 'trial8_best.pt'))
    if epoch%10==0 or epoch==1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Test: {test_rmse:.5f} | Best: {best_test:.5f} | {time.time()-start:.0f}s')

torch.save(best, os.path.join(SAVE_DIR, 'best_params.pt'))
print(f'\n학습 완료! 최고 Test RMSE: {best_test:.5f}')
print(f'학습 시간: {(time.time()-start)/60:.1f}분')

## 5단계: 최종 평가

In [ ]:
net.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'trial8_best.pt'), map_location=device))
net.eval()
print('모델 로드 완료')

def evaluate_trajectory_disp(net, data, window_size=1000):
    net.eval()
    gt_rot = data['gt_orientation']; gt_pos = data['gt_translation']
    seq_len = len(data['acc'])
    pred_world_disps = []
    with torch.no_grad():
        for start in range(0, seq_len - window_size, window_size):
            end = start + window_size
            acc = data['acc'][start:end].unsqueeze(0).to(device)
            gyro = data['gyro'][start:end].unsqueeze(0).to(device)
            motor = data['motor'][start:end].unsqueeze(0).to(device)
            rot = gt_rot[start:end].unsqueeze(0).to(device)
            rot_so3 = rot.Log().tensor().float()
            pred_d, _ = net(acc, gyro, rot_so3, motor)
            pred_d = pred_d.squeeze(0).cpu()
            for i in range(pred_d.shape[0]):
                t_idx = start + S_IDX + i * STRIDE
                t_next = t_idx + STRIDE
                if t_next < seq_len:
                    R_t = gt_rot[t_idx]
                    world_disp = R_t @ pred_d[i]
                    pred_world_disps.append((t_idx, t_next, world_disp))
    if len(pred_world_disps) < 2: return None
    pred_world_disps.sort(key=lambda x: x[0])
    first_t = pred_world_disps[0][0]
    pred_positions = {first_t: gt_pos[first_t].clone()}
    for t_start, t_next, disp in pred_world_disps:
        if t_start in pred_positions:
            pred_positions[t_next] = pred_positions[t_start] + disp
    valid_times = sorted(pred_positions.keys())
    if len(valid_times) < 10: return None
    pred_pos_arr = torch.stack([pred_positions[t] for t in valid_times]).numpy()
    gt_pos_arr = gt_pos[valid_times].numpy()
    pe = np.linalg.norm(pred_pos_arr - gt_pos_arr, axis=1)
    ate = float(np.sqrt(np.mean(pe**2)))
    dist = float(np.sum(np.linalg.norm(np.diff(gt_pos_arr, axis=0), axis=1)))
    tde = (ate/dist*100) if dist > 0 else 0
    return {'ATE': ate, 'TDE': tde, 'dist': dist} if np.isfinite(ate) else None

print(f'\n{"="*60}\n  SEEN 평가\n{"="*60}')
print(f'{"궤적":<15}{"ATE[m]":<12}{"TDE[%]":<10}')
print('-'*37)
seen_results = []
for i, d in enumerate(eval_data):
    name = SEEN[i].split('/')[0]
    r = evaluate_trajectory_disp(net, d)
    if r:
        print(f'  {name:<13}{r["ATE"]:>10.3f}  {r["TDE"]:>8.2f}')
        seen_results.append(r)
seen_avg = np.mean([r['ATE'] for r in seen_results]) if seen_results else float('nan')
print(f'  {"평균":<13}{seen_avg:>10.3f}')

print(f'\n{"="*60}\n  UNSEEN 평가\n{"="*60}')
print(f'{"궤적":<15}{"ATE[m]":<12}{"TDE[%]":<10}')
print('-'*37)
unseen_results = []
for name, d in unseen_data:
    r = evaluate_trajectory_disp(net, d)
    if r:
        print(f'  {name:<13}{r["ATE"]:>10.3f}  {r["TDE"]:>8.2f}')
        unseen_results.append(r)
unseen_avg = np.mean([r['ATE'] for r in unseen_results]) if unseen_results else float('nan')
print(f'  {"평균":<13}{unseen_avg:>10.3f}')

print(f'\n{"="*100}')
print(f'  모든 결과 비교')
print(f'{"="*100}')
print(f'{"":>30}{"SEEN":<15}{"UNSEEN":<15}')
print('-'*60)
print(f'  {"속도 예측 (Optuna)":<28}{0.523:>10.3f}     {2.263:>10.3f}')
print(f'  {"변위 baseline (속도 HP)":<28}{0.429:>10.3f}     {1.809:>10.3f}')
print(f'  {"변위 Trial 8 (30 epoch)":<28}{0.429:>10.3f}     {1.703:>10.3f}')
print(f'  {"변위 Trial 8 (100 epoch)":<28}{seen_avg:>10.3f}     {unseen_avg:>10.3f}    ← 이번 결과')
print('-'*60)
print(f'  {"AirIO+Motor (양방향)":<28}{0.460:>10.3f}     {0.767:>10.3f}    참고')
print('='*100)

if unseen_avg < 1.703:
    print(f'\n  ★ 30 epoch보다 개선!')
elif unseen_avg < 1.809:
    print(f'\n  baseline보다 좋음')
else:
    print(f'\n  baseline 비슷')

with open(os.path.join(SAVE_DIR, 'results.pkl'), 'wb') as f:
    pickle.dump({'seen_avg':seen_avg, 'unseen_avg':unseen_avg,
                 'seen_details':seen_results, 'unseen_details':unseen_results}, f)
print(f'\n결과 저장: {SAVE_DIR}/results.pkl')

## 6단계: 3D 궤적 애니메이션 (GIF)

선택한 궤적 1개에 대해 시간에 따라 Blackbird GT와 모델 예측이 함께 움직이는 경로를 GIF로 저장한다. `TYPE`/`FLIGHT_NUM`으로 궤적 선택, `N_FRAMES`/`FPS`로 길이·속도 조절.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from matplotlib.animation import FuncAnimation, PillowWriter
rcParams['font.family'] = 'sans-serif'
rcParams['axes.unicode_minus'] = False

# ======= 여기만 수정하세요 =======
TYPE = 'unseen'         # 'seen' 또는 'unseen'
FLIGHT_NUM = 1          # 궤적 번호 (1부터)

N_FRAMES = 120          # GIF 총 프레임 수 (적을수록 가볍고 빠름)
FPS = 20                # 초당 프레임
ROTATE = True           # True면 카메라가 천천히 회전

# z축 범위 (None이면 자동)
Z_MIN = None            # 예: 0
Z_MAX = None            # 예: 5
# ================================

# 모델 출력을 좌표로 환산해 (GT 궤적, 예측 궤적) 반환 (5단계와 동일 방식)
def predict_trajectory(net, data, window_size=1000):
    net.eval()
    gt_rot = data['gt_orientation']; gt_pos = data['gt_translation']
    seq_len = len(data['acc'])
    steps = []
    with torch.no_grad():
        for start in range(0, seq_len - window_size, window_size):
            end = start + window_size
            acc = data['acc'][start:end].unsqueeze(0).to(device)
            gyro = data['gyro'][start:end].unsqueeze(0).to(device)
            motor = data['motor'][start:end].unsqueeze(0).to(device)
            rot = gt_rot[start:end].unsqueeze(0).to(device)
            rot_so3 = rot.Log().tensor().float()
            pred_d, _ = net(acc, gyro, rot_so3, motor)
            pred_d = pred_d.squeeze(0).cpu()
            for i in range(pred_d.shape[0]):
                t_idx = start + S_IDX + i * STRIDE
                t_next = t_idx + STRIDE
                if t_next < seq_len:
                    steps.append((t_idx, t_next, gt_rot[t_idx] @ pred_d[i]))
    if len(steps) < 2:
        return None
    steps.sort(key=lambda x: x[0])
    first_t = steps[0][0]
    pred_pos = {first_t: gt_pos[first_t].clone()}
    for t0, t1, d in steps:
        if t0 in pred_pos:
            pred_pos[t1] = pred_pos[t0] + d
    ts = sorted(pred_pos.keys())
    if len(ts) < 10:
        return None
    pred_arr = torch.stack([pred_pos[t] for t in ts]).numpy()
    gt_arr = gt_pos[ts].numpy()
    return gt_arr, pred_arr

# 궤적 선택
if TYPE == 'unseen':
    items = [(name, d) for name, d in unseen_data]
else:
    items = [(SEEN[i].split('/')[0], d) for i, d in enumerate(eval_data)]
name, d = items[FLIGHT_NUM - 1]
gt, pred = predict_trajectory(net, d)
n = len(gt)
frames = np.linspace(0, n - 1, min(N_FRAMES, n)).astype(int)

# 그림 세팅
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

allpts = np.vstack([gt, pred])
ax.set_xlim(allpts[:, 0].min(), allpts[:, 0].max())
ax.set_ylim(allpts[:, 1].min(), allpts[:, 1].max())
zlo, zhi = allpts[:, 2].min(), allpts[:, 2].max()
if Z_MIN is not None: zlo = Z_MIN
if Z_MAX is not None: zhi = Z_MAX
ax.set_zlim(zlo, zhi)
ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_zlabel('Z [m]')

# 전체 경로를 흐리게 깔아두기
ax.plot(gt[:, 0], gt[:, 1], gt[:, 2], color='gray', lw=0.8, alpha=0.25)

gt_line,  = ax.plot([], [], [], 'k-', lw=2.2, label='Blackbird GT')
pred_line, = ax.plot([], [], [], 'r-', lw=1.8, label='Causal-MARIO')
gt_dot,   = ax.plot([], [], [], 'ko', ms=7)
pred_dot, = ax.plot([], [], [], 'ro', ms=6)
ax.legend(loc='upper left', fontsize=10)

def update(fi):
    i = frames[fi]
    gt_line.set_data(gt[:i+1, 0], gt[:i+1, 1]);   gt_line.set_3d_properties(gt[:i+1, 2])
    pred_line.set_data(pred[:i+1, 0], pred[:i+1, 1]); pred_line.set_3d_properties(pred[:i+1, 2])
    gt_dot.set_data([gt[i, 0]], [gt[i, 1]]);     gt_dot.set_3d_properties([gt[i, 2]])
    pred_dot.set_data([pred[i, 0]], [pred[i, 1]]); pred_dot.set_3d_properties([pred[i, 2]])
    ax.set_title(f'{TYPE.upper()} {name}  |  t = {i+1}/{n}', fontsize=13, fontweight='bold')
    if ROTATE:
        ax.view_init(elev=25, azim=-60 + 120 * fi / len(frames))
    return gt_line, pred_line, gt_dot, pred_dot

anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 / FPS, blit=False)

out_path = os.path.join(SAVE_DIR, f'traj_{TYPE}_{FLIGHT_NUM}.gif')
anim.save(out_path, writer=PillowWriter(fps=FPS), dpi=90)
plt.close(fig)
print(f'저장: {out_path}')

# 노트북에서 바로 재생
from IPython.display import Image
Image(filename=out_path)
